# 08 — B2 RadImageNet encoder preflight (short GPU run)

This is the **first gate** for the roadmap's owned RadImageNet model. It measures the exact frozen ResNet50 encoder on 64 studies from the existing 05A 224×224 uint8 cache. It does **not** train on the full dataset, use gold labels, create a submission, or change the 0.943 notebook.

First run **only the asset-check code cell** interactively with Accelerator **None**, Internet **Off**. It must print `ASSET CHECK PASSED` and list a cache with at least 4,349 `weak_*.npz` and 58 `gold_*.npz`. Then switch to **GPU T4×2** and use Save & Run All for the benchmark; only GPU 0 is used. The public RadImageNet weight input is pinned in notebook metadata. Additionally attach the saved 05A output or a private Dataset created from the old 05A version that actually contains `rsna_05a_dense_mil/study_cache`. A small artifacts ZIP, cache manifest alone, or output of notebook 07 is not the cache.

Use **Save & Run All** and send back the Output ZIP containing `b2_preflight.json`. Do not start a long training run based only on a successful import; this gate measures real cache decoding and encoder throughput first.


In [ ]:
from pathlib import Path
from time import monotonic
import gc, hashlib, json, os, random
import numpy as np
import torch
import torch.nn as nn
from torchvision.models import resnet50

CONFIG = {
    'experiment': 'B2_RadImageNet_frozen_encoder_preflight_v1',
    'seed': 111, 'sample_studies': 64, 'image_microbatch': 24,
    'max_preflight_seconds': 900,
    'expected_image_size': 224, 'expected_slices_max': 80,
    'encoder_sha256': '08629f7e7bd3e29b8ee9522ca3f65ce4d010a7ddf74f0ea3c7e3f3d0bbab0734',
}
OUT = Path('/kaggle/working/b2_preflight.json')
INPUT = Path('/kaggle/input')
if not INPUT.is_dir(): raise FileNotFoundError('Run this notebook on Kaggle')
random.seed(CONFIG['seed']); np.random.seed(CONFIG['seed']); torch.manual_seed(CONFIG['seed'])
torch.backends.cudnn.benchmark = False

def find_named_dir(name, max_depth=9):
    found = []
    for base, dirs, _ in os.walk(INPUT):
        depth = len(Path(base).relative_to(INPUT).parts)
        dirs[:] = [d for d in dirs if d not in {'train_series','test_series','dicom','dicoms'}]
        if name in dirs:
            found.append(Path(base) / name); dirs.remove(name)
        if depth >= max_depth: dirs[:] = []
    return found

cache_candidates = []
for d in find_named_dir('study_cache'):
    weak = sum(e.is_file() and e.name.startswith('weak_') and e.name.endswith('.npz') for e in os.scandir(d))
    gold = sum(e.is_file() and e.name.startswith('gold_') and e.name.endswith('.npz') for e in os.scandir(d))
    cache_candidates.append({'path': str(d), 'weak': weak, 'gold': gold})
print('CACHE CANDIDATES:', cache_candidates)
valid_caches = [r for r in cache_candidates if r['weak'] >= 4349 and r['gold'] >= 58]
if not valid_caches:
    raise FileNotFoundError('Attach the saved 05A Output version with the full study_cache (>=4349 weak, >=58 gold). No GPU work has started.')
cache = Path(sorted(valid_caches, key=lambda r: (-r['weak'], r['path']))[0]['path'])
weight_paths = [
    INPUT / 'datasets/marwanmath/resnet-50-radimagenet-marwan/ResNet50.pt',
    INPUT / 'resnet-50-radimagenet-marwan/ResNet50.pt',
]
weights = [p for p in weight_paths if p.is_file()]
if not weights:
    raise FileNotFoundError('Attach the public resnet-50-radimagenet-marwan dataset containing ResNet50.pt. No GPU work has started.')
weight = weights[0]
digest = hashlib.sha256()
with weight.open('rb') as fh:
    for block in iter(lambda: fh.read(8 << 20), b''): digest.update(block)
if digest.hexdigest() != CONFIG['encoder_sha256']:
    raise RuntimeError(f'RadImageNet weight SHA256 mismatch: {digest.hexdigest()}')
print('PINNED WEIGHT:', weight, 'SHA256:', digest.hexdigest())
print('CACHE:', cache)
print('ASSET CHECK PASSED: 05A cache and pinned RadImageNet weights are available')
Path('/kaggle/working/b2_assets.json').write_text(json.dumps({
    'status': 'READY', 'cache': str(cache), 'cache_candidates': cache_candidates,
    'weight': str(weight), 'weight_sha256': digest.hexdigest()
}, indent=2), encoding='utf-8')


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError('Asset check passed. Select GPU T4x2 to run the encoder benchmark.')
if 'T4' not in torch.cuda.get_device_name(0):
    raise RuntimeError(f'Expected T4 on GPU 0, got {torch.cuda.get_device_name(0)}')
print('DEVICE:', torch.cuda.get_device_name(0))

class RadEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = nn.Sequential(*list(resnet50(weights=None).children())[:-2])
    def forward(self, x):
        return self.backbone(x).mean(dim=(2, 3))

load_start = monotonic()
model = RadEncoder()
state = torch.load(weight, map_location='cpu', weights_only=True)
expected = model.state_dict()
missing = sorted(set(expected) - set(state)); extra = sorted(set(state) - set(expected))
bad_shapes = sorted(k for k in set(expected) & set(state) if tuple(expected[k].shape) != tuple(state[k].shape))
if missing or extra or bad_shapes:
    raise RuntimeError(f'Encoder state mismatch: missing={missing[:5]}, extra={extra[:5]}, shapes={bad_shapes[:5]}')
model.load_state_dict(state, strict=True)
del state; gc.collect()
model = model.to('cuda:0').eval().requires_grad_(False)
torch.cuda.synchronize(0)
load_seconds = monotonic() - load_start
print(f'Encoder loaded strictly in {load_seconds:.1f}s; parameters={sum(p.numel() for p in model.parameters()):,}')

all_files = sorted(e.path for e in os.scandir(cache) if e.is_file() and e.name.startswith('weak_') and e.name.endswith('.npz'))
if len(all_files) < 4349: raise RuntimeError('Full weak cache disappeared after asset check')
indices = np.linspace(0, len(all_files)-1, CONFIG['sample_studies'], dtype=int)
chosen = [all_files[int(i)] for i in indices]
n_images = 0; n_empty = 0; processed = 0; slice_counts = []; checksum = 0.0
torch.cuda.reset_peak_memory_stats(0)
start = monotonic()
with torch.inference_mode():
    for study_i, path in enumerate(chosen, 1):
        with np.load(path, allow_pickle=False) as z:
            if not {'images','slot_ids','slice_indices','positions'}.issubset(z.files):
                raise RuntimeError(f'Invalid cache schema: {path}')
            images = z['images']
            if images.dtype != np.uint8 or images.ndim != 3 or images.shape[1:] != (224,224):
                raise RuntimeError(f'Invalid cached image shape/dtype: {path}, {images.shape}, {images.dtype}')
            if len(images) > CONFIG['expected_slices_max']:
                raise RuntimeError(f'Too many slices for expected 05A cache: {path}, {len(images)}')
            if not (len(z['slot_ids']) == len(z['slice_indices']) == len(z['positions']) == len(images)):
                raise RuntimeError(f'Cache metadata length mismatch: {path}')
        if len(images) == 0: n_empty += 1
        slice_counts.append(len(images)); n_images += len(images)
        for offset in range(0, len(images), CONFIG['image_microbatch']):
            part = images[offset:offset + CONFIG['image_microbatch']]
            gray = torch.from_numpy(part).to('cuda:0').float().div_(127.5).sub_(1.0)
            rgb = gray.unsqueeze(1).expand(-1, 3, -1, -1).contiguous()
            with torch.autocast('cuda', dtype=torch.float16):
                feat = model(rgb)
            if feat.shape != (len(part), 2048) or not bool(torch.isfinite(feat).all()):
                raise RuntimeError(f'Nonfinite or wrong feature shape in {path}')
            checksum += float(feat.float().mean().item())
            del gray, rgb, feat
        del images
        processed = study_i
        if study_i % 16 == 0:
            torch.cuda.synchronize(0)
            print(f'encoded {study_i}/{len(chosen)} studies; images={n_images}; elapsed={monotonic()-start:.1f}s', flush=True)
        if monotonic() - start >= CONFIG['max_preflight_seconds']:
            print('TIME GUARD: stopped the benchmark and will save a partial report')
            break
torch.cuda.synchronize(0)
elapsed = monotonic() - start
peak_alloc = torch.cuda.max_memory_allocated(0) / 2**30
peak_reserved = torch.cuda.max_memory_reserved(0) / 2**30
report = {
    'status': 'COMPLETE' if processed == len(chosen) else 'PARTIAL_TIME_GUARD',
    'config': CONFIG, 'cache': str(cache), 'cache_candidates': cache_candidates,
    'weight': str(weight), 'weight_sha256': digest.hexdigest(),
    'device': torch.cuda.get_device_name(0), 'sample_studies': processed,
    'images': n_images, 'empty_studies': n_empty,
    'mean_slices_per_study': float(np.mean(slice_counts)),
    'max_slices_per_study': int(np.max(slice_counts)),
    'weight_load_seconds': load_seconds, 'encoder_seconds': elapsed,
    'images_per_second': n_images / elapsed,
    'rough_full_4407_hours_one_T4': elapsed * 4407 / processed / 3600,
    'cuda_peak_allocated_gib': peak_alloc, 'cuda_peak_reserved_gib': peak_reserved,
    'feature_checksum_nonclinical': checksum,
    'caveat': '64-study one-T4 sample estimate; does not include full feature writes, head training, or inference',
}
OUT.write_text(json.dumps(report, indent=2), encoding='utf-8')
print(json.dumps({k: report[k] for k in ['status','sample_studies','images','encoder_seconds','rough_full_4407_hours_one_T4','cuda_peak_allocated_gib','cuda_peak_reserved_gib']}, indent=2))
print('No full training was run. Send b2_preflight.json before enabling a long experiment.')
